### Partie 2 : Nettoyage avance avec Pandas (3-4h)

**Competence evaluee : C2.2 - Traiter des donnees structurees avec un langage de programmation**

#### Etape 2.1 : Nettoyage des donnees meteo
- Charger `meteo_raw.csv` avec Pandas
- Standardiser les formats de dates
- Convertir les colonnes numeriques en gerant les erreurs
- Corriger les valeurs aberrantes :
  - Temperatures hors [-40, 50] -> NaN puis interpolation
  - Humidite hors [0, 100] -> clipping
  - Rayonnement solaire negatif -> 0
- Traiter les valeurs manquantes :
  - Interpolation lineaire pour temperature et humidite
  - Forward fill pour precipitation
- Ajouter des colonnes temporelles (jour, mois, saison, jour de semaine)

**Livrables** :
- Notebook `04_nettoyage_meteo_pandas.ipynb`
- Dataset nettoye `output/meteo_clean.csv`
- Rapport avant/apres nettoyage (completude par colonne)

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Chemins
DATA_DIR = "../data"
OUTPUT_DIR = "../output"

In [3]:
df_weather_raw = pd.read_csv(f"{DATA_DIR}/meteo_raw.csv")

print(f"Shape: {df_weather_raw.shape}")
print(f"\nColonnes: {df_weather_raw.columns.tolist()}")
df_weather_raw.head(10)

Shape: (252612, 7)

Colonnes: ['commune', 'timestamp', 'temperature_c', 'humidite_pct', 'rayonnement_solaire_wm2', 'vitesse_vent_kmh', 'precipitation_mm']


,commune,timestamp,temperature_c,humidite_pct,rayonnement_solaire_wm2,vitesse_vent_kmh,precipitation_mm
0,Saint-Etienne,09/15/2024 15:00:00,17.1,143.3,244.9,14.3,0.0
1,Bordeaux,21/07/2023 15:00,19.6,50.6,414.9,3.2,0.0
2,Montpellier,2023-09-18 20:00:00,18.3,65.7,218.4,13.6,0.0
3,Le Havre,01/03/2024 22:00:00,3.7,94.9,6.8,18.6,11.6
4,Lille,29/10/2024 20:00,14.0,42.9,781.8,4.0,0.0
5,Bordeaux,22/12/2023 13:00,4.4,36.9,796.4,6.1,0.0
6,Marseille,09/15/2023 21:00:00,22.5,86.8,5.8,32.6,0.0
7,Toulouse,30/05/2023 00:00,8.3,66.3,26.4,31.4,7.2
8,Bordeaux,2024-10-05T09:00:00,11.5,69.8,71.4,34.4,0.0
9,Toulon,2024-09-28T21:00:00,19.2,79.0,13.1,31.8,0.0


Standardiser les formats de dates

In [4]:
def parse_timestamp(ts):
    """Parse les timestamps multi-formats."""
    if pd.isna(ts):
        return pd.NaT
    
    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S",
    ]
    
    for fmt in formats:
        try:
            return datetime.strptime(str(ts), fmt)
        except ValueError:
            continue
    
    return pd.NaT

df_weather = df_weather_raw.copy()

# 1. Parser les timestamps
print("[1/5] Parsing des timestamps...")
df_weather['timestamp'] = df_weather['timestamp'].apply(parse_timestamp)
invalid_ts = df_weather['timestamp'].isna().sum()
print(f"  Timestamps invalides: {invalid_ts}")

# Supprimer les lignes sans timestamp valide
df_weather = df_weather.dropna(subset=['timestamp'])

[1/5] Parsing des timestamps...
  Timestamps invalides: 0


In [ ]:
# 2. Convertir les colonnes numeriques
print("\n[2/5] Conversion des colonnes numeriques...")

# Temperature (remplacer virgule par point)
df_weather['temperature_c'] = pd.to_numeric(
    df_weather['temperature_c'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

# rayonnement (remplacer virgule par point)
df_weather['rayonnement_solaire_wm2'] = pd.to_numeric(
    df_weather['rayonnement_solaire_wm2'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

# Autres colonnes
df_weather['humidite_pct'] = pd.to_numeric(df_weather['humidite_pct'], errors='coerce')
df_weather['vitesse_vent_kmh'] = pd.to_numeric(df_weather['vitesse_vent_kmh'], errors='coerce')
df_weather['precipitation_mm'] = pd.to_numeric(df_weather['precipitation_mm'], errors='coerce')

print("  Conversions effectuees.")


[2/5] Conversion des colonnes numeriques...
  Conversions effectuees.


In [8]:
# 3. Corriger les valeurs aberrantes
print("\n[3/5] Correction des valeurs aberrantes...")

# Temperatures hors [-40, 50] -> NaN
temp_outliers = ((df_weather['temperature_c'] < -40) | (df_weather['temperature_c'] > 50)).sum()
df_weather.loc[
    (df_weather['temperature_c'] < -40) | (df_weather['temperature_c'] > 50),
    'temperature_c'
] = np.nan
print(f"  Temperatures aberrantes -> NaN: {temp_outliers}")

# Humidite hors [0, 100] -> clipper
humidity_outliers = ((df_weather['humidite_pct'] < 0) | (df_weather['humidite_pct'] > 100)).sum()
df_weather['humidite_pct'] = df_weather['humidite_pct'].clip(0, 100)
print(f"  Humidite clippee [0, 100]: {humidity_outliers}")


[3/5] Correction des valeurs aberrantes...
  Temperatures aberrantes -> NaN: 1977
  Humidite clippee [0, 100]: 1801


In [ ]:
print("\n Correction du rayonnement")  #- Rayonnement solaire negatif -> 0

# rayonnement >0 -> NaN
beam_outliers = ((df_weather['rayonnement_solaire_wm2'] < 0)).sum()
df_weather.loc[
    (df_weather['rayonnement_solaire_wm2'] < 0),
    'rayonnement_solaire_wm2'
] = np.nan
print(f"  Rayonnement -> NaN: {temp_outliers}")



[4/5] Correction du rayonnement
  Rayonnement -> NaN: 1977


In [25]:

# 4. Traiter les valeurs manquantes
print("\n[4/5] Traitement des valeurs manquantes...")

# Trier par ville et timestamp pour l'interpolation
df_weather = df_weather.sort_values(['commune', 'timestamp'])

# Interpolation lineaire pour temperature, rayonnement et vitesse-vent (par ville)
for col in ['temperature_c', 'humidite_pct', 'rayonnement_solaire_wm2', 'vitesse_vent_kmh']:
    before_na = df_weather[col].isna().sum()
    df_weather[col] = df_weather.groupby('commune')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    after_na = df_weather[col].isna().sum()
    print(f"  {col}: {before_na} -> {after_na} NaN (interpoles: {before_na - after_na})")

# Forward fill pour weather_condition
before_na = (df_weather['precipitation_mm'].isna() | (df_weather['precipitation_mm'] == '')).sum()
df_weather['precipitation_mm'] = df_weather['precipitation_mm'].replace('', np.nan)
df_weather['precipitation_mm'] = df_weather.groupby('commune')['precipitation_mm'].transform(
    lambda x: x.ffill().bfill()
)
after_na = df_weather['precipitation_mm'].isna().sum()
print(f"  precipitation_mm: {before_na} -> {after_na} NaN (forward filled)")


[4/5] Traitement des valeurs manquantes...
  temperature_c: 0 -> 0 NaN (interpoles: 0)
  humidite_pct: 0 -> 0 NaN (interpoles: 0)
  rayonnement_solaire_wm2: 0 -> 0 NaN (interpoles: 0)
  vitesse_vent_kmh: 0 -> 0 NaN (interpoles: 0)
  precipitation_mm: 0 -> 0 NaN (forward filled)


In [32]:
# 5. Ajouter des colonnes temporelles
print("\n[5/5] Ajout des colonnes temporelles...")

df_weather['date'] = df_weather['timestamp'].dt.date
df_weather['day_of_week'] = df_weather['timestamp'].dt.dayofweek
df_weather['month'] = df_weather['timestamp'].dt.month
df_weather['season'] = df_weather['month'].map({
    12: 'Hiver', 1: 'Hiver', 2: 'Hiver',
    3: 'Printemps', 4: 'Printemps', 5: 'Printemps',
    6: 'Ete', 7: 'Ete', 8: 'Ete',
    9: 'Automne', 10: 'Automne', 11: 'Automne'
})

print("  Colonnes ajoutees: date, hour, day_of_week, month, season")


[5/5] Ajout des colonnes temporelles...
  Colonnes ajoutees: date, hour, day_of_week, month, season


In [33]:
def quality_report(df, name="DataFrame"):
    print(f"RAPPORT QUALITE - {name}")
    print(f"Lignes: {len(df):,}")
    print(f"Colonnes: {len(df.columns)}")
    
    report = []
    for col in df.columns:
        total = len(df)
        missing = df[col].isna().sum() + (df[col] == '').sum() if df[col].dtype == 'object' else df[col].isna().sum()
        completude = (1 - missing / total) * 100
        unique = df[col].nunique()
        dtype = df[col].dtype
        
        report.append({
            'Colonne': col,
            'Type': str(dtype),
            'Manquants': missing,
            'Completude %': round(completude, 2),
            'Uniques': unique
        })
    
    return pd.DataFrame(report)

raw_report = quality_report(df_weather_raw, "Weather Raw")
print(raw_report)

print("--------------------")

cleaned_report = quality_report(df_weather, "Weather Clean")
print(cleaned_report)

RAPPORT QUALITE - Weather Raw
Lignes: 252,612
Colonnes: 7
                   Colonne     Type  Manquants  Completude %  Uniques
0                  commune      str          0        100.00       15
1                timestamp      str          0        100.00    69066
2            temperature_c      str       1229         99.51      810
3             humidite_pct  float64          0        100.00     1094
4  rayonnement_solaire_wm2  float64          0        100.00     8685
5         vitesse_vent_kmh  float64          0        100.00      401
6         precipitation_mm  float64          0        100.00      151
--------------------
RAPPORT QUALITE - Weather Clean
Lignes: 252,612
Colonnes: 12
                    Colonne            Type  Manquants  Completude %  Uniques
0                   commune             str          0         100.0       15
1                 timestamp  datetime64[us]          0         100.0    17544
2             temperature_c         float64          0         100

### Export csv

In [34]:
df_weather.to_csv(f"{OUTPUT_DIR}/meteo_clean.csv", index=False)